# NB04 — Independent Training Runs: ViT-Base/16

Trains ViT-Base/16 from scratch, independently, once per random seed, on the
specimen-disjoint split from NB01, then evaluates each run on the untouched
`common_test` set. This yields genuinely independent observations, which is the
precondition the Friedman and Wilcoxon significance tests (run in NB06) require.
Re-scoring a single trained checkpoint across resampled partitions of its own
training/validation pool does not meet that precondition, since the resulting scores
are correlated through shared training data rather than independent draws.

Report these runs as **independent training runs**, never as cross-validation folds —
NB06 is where the significance tests are computed. ViT-Base/16 is isolated in its own
notebook (rather than grouped with NB03's CNNs) because it is the heaviest model here
and needs its own 12-hour session.

**Model:** ViT-Base/16.
**Runtime:** ≈ 8–10 h. Increasing `SEEDS` increases statistical power if compute budget allows.


In [1]:
# ===== Imports =====
import os, json, time, random, warnings, math
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import timm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


device: cuda | Tesla T4


In [2]:
# ===== Config =====
RAW_DATASET_DIR = '/kaggle/input/datasets/maruf170102/bdlithi/Dataset'   # <-- EDIT: folder containing the 11 class sub-folders
SPLITS_DIR = Path('/kaggle/input/datasets/maruf170102/bdlitchi-revision-splits/revision_splits')   # NB01 output
OUT_DIR    = Path('/kaggle/working/revision_seeds_vit'); OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR   = OUT_DIR/'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)

MODELS   = ['ViTBase16']
SEEDS    = [42, 123, 7]      # each seed = one fully independent training run
EPOCHS   = 30
PATIENCE = 7
SAVE_CHECKPOINTS = True      # set False if disk is tight

In [3]:
def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [4]:
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]

class LitchiDataset(Dataset):
    def __init__(self, df, transform, label2idx, grayscale=False):
        self.df=df.reset_index(drop=True); self.t=transform
        self.l2i=label2idx; self.gray=grayscale
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; img=Image.open(r.filepath).convert('RGB')
        if self.gray:
            img = img.convert('L').convert('RGB')   # drop colour, keep 3 channels
        return self.t(img), self.l2i[r.label]

def make_transforms(sz, augment):
    if augment:
        tr=transforms.Compose([transforms.Resize((sz+32,sz+32)), transforms.RandomCrop(sz),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.3,0.3,0.2,0.05), transforms.RandomRotation(20),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    else:
        tr=transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    ev=transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])
    return tr,ev

def build_backbone(name, nc, pretrained=True):
    if name=='ResNet50':
        m=torchvision.models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
        m.fc=nn.Linear(m.fc.in_features,nc); return m,224,32
    if name=='MobileNetV3Large':
        m=torchvision.models.mobilenet_v3_large(weights='IMAGENET1K_V2' if pretrained else None)
        m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,nc); return m,224,32
    if name=='EfficientNetB0':
        return timm.create_model('efficientnet_b0',pretrained=pretrained,num_classes=nc),224,32
    if name=='EfficientNetB3':
        return timm.create_model('efficientnet_b3',pretrained=pretrained,num_classes=nc),300,16
    if name=='ViTBase16':
        return timm.create_model('vit_base_patch16_224',pretrained=pretrained,num_classes=nc),224,16
    raise ValueError(name)


In [5]:
def train_model(name, trn_df, val_df, label2idx, seed, epochs=30, patience=7,
                lr=3e-4, wd=1e-4, grayscale=False, log_every=1):
    """Independent training run. Returns (model, history, best_epoch)."""
    set_all_seeds(seed)
    nc=len(label2idx)
    model,sz,bs = build_backbone(name,nc); model=model.to(device)
    ttf,etf = make_transforms(sz, augment=True)
    trn=DataLoader(LitchiDataset(trn_df,ttf,label2idx,grayscale), batch_size=bs, shuffle=True,
                   num_workers=2, pin_memory=True, drop_last=False)
    val=DataLoader(LitchiDataset(val_df,etf,label2idx,grayscale), batch_size=bs, shuffle=False,
                   num_workers=2, pin_memory=True)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    crit=nn.CrossEntropyLoss()
    scaler=torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
    hist={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    best=float('inf'); best_state=None; best_ep=-1; bad=0
    for ep in range(epochs):
        model.train(); rl=rc=rt=0
        for x,y in trn:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                o=model(x); loss=crit(o,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            rl+=loss.item()*x.size(0); rc+=(o.argmax(1)==y).sum().item(); rt+=x.size(0)
        tl,ta=rl/rt,rc/rt
        model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for x,y in val:
                x,y=x.to(device),y.to(device)
                with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
                    o=model(x); loss=crit(o,y)
                vl+=loss.item()*x.size(0); vc+=(o.argmax(1)==y).sum().item(); vt+=x.size(0)
        vl,va=vl/vt,vc/vt
        sch.step()
        hist['train_loss'].append(tl); hist['val_loss'].append(vl)
        hist['train_acc'].append(ta);  hist['val_acc'].append(va)
        if log_every and (ep+1)%log_every==0:
            print(f'  ep {ep+1:02d}/{epochs} train_loss={tl:.4f} acc={ta:.4f} | val_loss={vl:.4f} acc={va:.4f}')
        if vl<best-1e-5:
            best=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_ep=ep+1; bad=0
        else:
            bad+=1
            if bad>=patience:
                print(f'  early stop @ epoch {ep+1} (best epoch {best_ep})'); break
    model.load_state_dict(best_state)
    return model, hist, best_ep

@torch.no_grad()
def predict(model, df, label2idx, img_size, bs=32, grayscale=False):
    _,etf=make_transforms(img_size,False)
    ld=DataLoader(LitchiDataset(df,etf,label2idx,grayscale),batch_size=bs,shuffle=False,num_workers=2)
    model.eval(); P=[];Y=[];B=[]
    for x,y in ld:
        p=F.softmax(model(x.to(device)),1).cpu().numpy()
        P.extend(p.argmax(1)); Y.extend(y.numpy()); B.extend(p)
    return np.array(Y),np.array(P),np.array(B)

def metrics_dict(y,p,prob,nc):
    d=dict(accuracy=accuracy_score(y,p), balanced_accuracy=balanced_accuracy_score(y,p),
           f1_macro=f1_score(y,p,average='macro',zero_division=0),
           f1_weighted=f1_score(y,p,average='weighted',zero_division=0),
           precision_macro=precision_score(y,p,average='macro',zero_division=0),
           recall_macro=recall_score(y,p,average='macro',zero_division=0),
           mcc=matthews_corrcoef(y,p))
    try: d['roc_auc_macro']=roc_auc_score(np.eye(nc)[y],prob,average='macro',multi_class='ovr')
    except Exception: d['roc_auc_macro']=float('nan')
    return d


In [6]:
# ===== Artifact saving: predictions, probabilities, histories, calibration =====
def expected_calibration_error(y_true, y_prob, n_bins=15):
    conf = y_prob.max(1); pred = y_prob.argmax(1); acc = (pred==y_true).astype(float)
    bins=np.linspace(0,1,n_bins+1); ece=0.0; rows=[]
    for i in range(n_bins):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()==0: rows.append((float(bins[i]),float(bins[i+1]),0,np.nan,np.nan)); continue
        a,c=acc[m].mean(),conf[m].mean()
        ece+=(m.sum()/len(conf))*abs(a-c)
        rows.append((float(bins[i]),float(bins[i+1]),int(m.sum()),float(a),float(c)))
    return float(ece), rows

def save_run_artifacts(out_dir, tag, y_true, y_pred, y_prob, history=None,
                       classes=None, extra=None, specimen_ids=None, filenames=None):
    """Everything NB06 needs, with no re-inference.

    specimen_ids is essential: NB06 bootstraps by SPECIMEN, not by image. Images inside
    one specimen are the same leaf and are strongly correlated, so an image-level
    bootstrap would badly understate the confidence interval.
    """
    pdir=Path(out_dir)/'predictions'; pdir.mkdir(parents=True, exist_ok=True)
    payload=dict(y_true=np.asarray(y_true), y_pred=np.asarray(y_pred), y_prob=np.asarray(y_prob))
    if specimen_ids is not None:
        payload['specimen_id']=np.asarray(specimen_ids).astype(str)
    if filenames is not None:
        payload['filename']=np.asarray(filenames).astype(str)
    np.savez_compressed(pdir/f'{tag}.npz', **payload)
    if history is not None:
        hdir=Path(out_dir)/'histories'; hdir.mkdir(parents=True, exist_ok=True)
        json.dump(history, open(hdir/f'{tag}.json','w'))
    ece,bins = expected_calibration_error(np.asarray(y_true), np.asarray(y_prob))
    cdir=Path(out_dir)/'calibration'; cdir.mkdir(parents=True, exist_ok=True)
    json.dump({'tag':tag,'ece':ece,'bins':bins,'classes':classes,**(extra or {})},
              open(cdir/f'{tag}.json','w'), indent=2)
    return ece


In [7]:
# ===== Load splits =====
man=pd.read_csv(SPLITS_DIR/'manifest_full.csv')
# ===== Rebuild image paths for THIS session =====
# manifest_full.csv stores absolute paths from the machine that ran NB01. Kaggle mounts
# datasets at different locations per session/account, so those paths are not portable.
# Rebuild them from RAW_DATASET_DIR + label + filename, then verify every file exists.
man['filepath'] = [str(Path(RAW_DATASET_DIR)/l/f) for l,f in zip(man.label, man.filename)]
missing = [p for p in man.filepath if not Path(p).exists()]
if missing:
    print(f'*** {len(missing)} of {len(man)} images NOT found. First few:')
    for p in missing[:5]: print('   ', p)
    print('\nFix RAW_DATASET_DIR above. It must be the folder that directly contains the')
    print('11 class sub-folders. Check the exact mount with:')
    print("   import os; print(os.listdir('/kaggle/input'))")
    raise FileNotFoundError(f'{len(missing)} images missing - stop here, do not train.')
print(f'OK - all {len(man)} image paths resolve in this session.')

CLASSES=sorted(man.label.unique()); NC=len(CLASSES)
label2idx={c:i for i,c in enumerate(CLASSES)}
pool=man[man.reserve=='pool']
trn=pool[pool.group_split=='train']; val=pool[pool.group_split=='val']
common_test=man[man.reserve=='common_test']
print(f'train={len(trn)} val={len(val)} common_test={len(common_test)} classes={NC}')
assert not (set(trn.filename)&set(common_test.filename)), 'train/common_test overlap!'


OK - all 11094 image paths resolve in this session.
train=6625 val=1397 common_test=1665 classes=11


In [8]:
# ===== Independent runs =====
records=[]; per_class={}
for arch in MODELS:
    for seed in SEEDS:
        tag=f'{arch}_seed{seed}'
        print(f'\n{"="*64}\n{tag}\n{"="*64}')
        t0=time.time()
        model,hist,best_ep=train_model(arch,trn,val,label2idx,seed=seed,
                                       epochs=EPOCHS,patience=PATIENCE)
        mins=(time.time()-t0)/60
        _,img_size,bs=build_backbone(arch,NC,pretrained=False)
        y,p,pr=predict(model,common_test,label2idx,img_size,bs)
        m=metrics_dict(y,p,pr,NC)
        m.update({'model':arch,'seed':seed,'best_epoch':best_ep,'train_minutes':round(mins,1)})
        records.append(m)
        per_class[tag]=f1_score(y,p,average=None,labels=list(range(NC)),zero_division=0).tolist()
        # persist predictions + probabilities + convergence history + calibration
        m['ece']=save_run_artifacts(OUT_DIR, tag, y, p, pr, history=hist,
                                    classes=CLASSES, extra={'set':'common_test',
                                    'model':arch,'seed':seed},
                                    specimen_ids=common_test.specimen_id.values,
                                    filenames=common_test.filename.values)
        if SAVE_CHECKPOINTS:
            torch.save(model.state_dict(), CKPT_DIR/f'{tag}.pt')
        # save incrementally so a session timeout does not lose everything
        pd.DataFrame(records).to_csv(OUT_DIR/'independent_runs.csv',index=False)
        json.dump(per_class,open(OUT_DIR/'per_class_f1_runs.json','w'),indent=2)
        json.dump({'classes':CLASSES},open(OUT_DIR/'classes.json','w'))
        print(f'  common_test acc={m["accuracy"]:.4f} f1={m["f1_macro"]:.4f} mcc={m["mcc"]:.4f} ({mins:.1f} min)')
        del model; torch.cuda.empty_cache()

res=pd.DataFrame(records)
print('\n=== per-run results on common_test ===')
print(res[['model','seed','accuracy','f1_macro','mcc','train_minutes']].round(4).to_string(index=False))
print('\n=== mean +/- std across independent runs ===')
print(res.groupby('model')[['accuracy','f1_macro','mcc']].agg(['mean','std']).round(4))
print('\nSaved to',OUT_DIR,'- save as a Kaggle Dataset for NB06.')


ViTBase16_seed42


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  ep 01/30 train_loss=1.2976 acc=0.5422 | val_loss=0.9593 acc=0.7008
  ep 02/30 train_loss=0.5016 acc=0.8296 | val_loss=0.6384 acc=0.7938
  ep 03/30 train_loss=0.3697 acc=0.8779 | val_loss=0.6562 acc=0.8046
  ep 04/30 train_loss=0.2900 acc=0.9052 | val_loss=0.4933 acc=0.8239
  ep 05/30 train_loss=0.2735 acc=0.9084 | val_loss=0.4392 acc=0.8590
  ep 06/30 train_loss=0.2428 acc=0.9212 | val_loss=0.5895 acc=0.8339
  ep 07/30 train_loss=0.2219 acc=0.9236 | val_loss=0.6683 acc=0.8082
  ep 08/30 train_loss=0.2086 acc=0.9313 | val_loss=0.4177 acc=0.8697
  ep 09/30 train_loss=0.1815 acc=0.9420 | val_loss=0.4355 acc=0.8525
  ep 10/30 train_loss=0.1694 acc=0.9452 | val_loss=0.4779 acc=0.8805
  ep 11/30 train_loss=0.1612 acc=0.9467 | val_loss=0.3330 acc=0.9026
  ep 12/30 train_loss=0.1554 acc=0.9526 | val_loss=0.4376 acc=0.8790
  ep 13/30 train_loss=0.1394 acc=0.9562 | val_loss=0.3274 acc=0.9019
  ep 14/30 train_loss=0.1126 acc=0.9630 | val_loss=0.3061 acc=0.9105
  ep 15/30 train_loss=0.0998 acc=0